In [1]:
from google.colab import files

uploaded = files.upload()

Saving agent_transactions.csv to agent_transactions (1).csv
Saving agents.csv to agents (1).csv
Saving cleaning_pipeline.py to cleaning_pipeline (1).py


In [2]:
import pandas as pd
import numpy as np

agents = pd.read_csv("agents.csv")

txns = pd.read_csv("agent_transactions.csv")

print("Agents shape:", agents.shape)
print("Transactions shape:", txns.shape)

display(agents.head())
display(txns.head())

Agents shape: (20, 3)
Transactions shape: (612, 5)


,agent_id,county,onboarded
0,AG001,Mombasa,2024-01-01
1,AG002,Nairobi,2024-02-01
2,AG003,Kisumu,2024-03-01
3,AG004,Kiambu,2024-04-01
4,AG005,Mombasa,2024-05-01


,txn_id,agent_id,txn_type,amount,txn_date
0,TX00001,AG009,deposit,353.0,2026-03-03
1,TX00002,AG020,deposit,1238.0,2026-04-27
2,TX00003,AG008,DEPOSIT,9129.0,2026-04-29
3,TX00004,AG009,deposit,652.0,2026-04-24
4,TX00005,AG009,withdrawal,478.0,2026-04-17


In [3]:
print("AGENT DATA TYPES")
print(agents.dtypes)

print("\nTRANSACTION DATA TYPES")
print(txns.dtypes)

print("\nMISSING VALUES")
print(txns.isna().sum())

print("\nTRANSACTION TYPES")
print(txns["txn_type"].value_counts(dropna=False))

print("\nDUPLICATE TRANSACTION IDs")
print(txns["txn_id"].duplicated().sum())

print("\nUNMATCHED AGENT IDs")
print((~txns["agent_id"].isin(agents["agent_id"])).sum())

AGENT DATA TYPES
agent_id     object
county       object
onboarded    object
dtype: object

TRANSACTION DATA TYPES
txn_id       object
agent_id     object
txn_type     object
amount      float64
txn_date     object
dtype: object

MISSING VALUES
txn_id       0
agent_id     0
txn_type     0
amount      22
txn_date     0
dtype: int64

TRANSACTION TYPES
txn_type
deposit       142
withdrawal    131
Deposit       128
DEPOSIT       111
Withdrawal    100
Name: count, dtype: int64

DUPLICATE TRANSACTION IDs
12

UNMATCHED AGENT IDs
8


In [4]:
# Check inconsistent date formats
print("DATES CONTAINING SLASHES:")
print(txns["txn_date"].str.contains("/", na=False).sum())

# Display examples
print(txns.loc[
    txns["txn_date"].str.contains("/", na=False),
    ["txn_id", "txn_date"]
].head())

# Check invalid numeric amounts
numeric_amounts = pd.to_numeric(txns["amount"], errors="coerce")

invalid_amounts = (
    txns["amount"].notna() & numeric_amounts.isna()
)

print("\nINVALID NUMERIC AMOUNTS:")
print(invalid_amounts.sum())

# Investigate the KES 250,000 values
print("\nKES 250,000 VALUES:")
print(txns["amount"].eq(250000).sum())

# Check agent onboarding date format
print("\nAGENT ONBOARDING DATE TYPE:")
print(agents["onboarded"].dtype)

DATES CONTAINING SLASHES:
5
      txn_id    txn_date
87   TX00088  2026/04/27
128  TX00129  2026/04/28
181  TX00182  2026/03/20
437  TX00438  2026/03/30
571  TX00572  2026/03/04

INVALID NUMERIC AMOUNTS:
0

KES 250,000 VALUES:
5

AGENT ONBOARDING DATE TYPE:
object


In [5]:
audit_data = [
    {
        "issue": "Duplicate transaction IDs",
        "dimension": "Uniqueness",
        "column": "txn_id",
        "rows_affected": 12
    },
    {
        "issue": "Missing or N/A transaction amounts",
        "dimension": "Completeness",
        "column": "amount",
        "rows_affected": 22
    },
    {
        "issue": "Inconsistent transaction type labels",
        "dimension": "Consistency",
        "column": "txn_type",
        "rows_affected": 384
    },
    {
        "issue": "Slash-formatted transaction dates",
        "dimension": "Validity",
        "column": "txn_date",
        "rows_affected": 5
    },
    {
        "issue": "Transactions referencing unknown agents",
        "dimension": "Referential integrity",
        "column": "agent_id",
        "rows_affected": 8
    },
    {
        "issue": "Potentially suspicious KES 250,000 amounts",
        "dimension": "Validity",
        "column": "amount",
        "rows_affected": 5
    }
]

audit_df = pd.DataFrame(audit_data)

display(audit_df)

,issue,dimension,column,rows_affected
0,Duplicate transaction IDs,Uniqueness,txn_id,12
1,Missing or N/A transaction amounts,Completeness,amount,22
2,Inconsistent transaction type labels,Consistency,txn_type,384
3,Slash-formatted transaction dates,Validity,txn_date,5
4,Transactions referencing unknown agents,Referential integrity,agent_id,8
5,"Potentially suspicious KES 250,000 amounts",Validity,amount,5


**Audit note:** The initial audit identified 8 transactions referencing unknown agents before duplicate removal. After removing 12 duplicate transaction IDs, 7 transactions referencing unknown agent AG099 remained during the merge validation. The difference is due to duplicate removal.

### Data Quality Audit Findings

The initial audit identified six data-quality concerns in the mobile money transaction dataset. These included 12 duplicate transaction IDs, 22 missing or N/A amounts, 384 inconsistent transaction type labels, five slash-formatted transaction dates, eight transactions referencing unknown agents, and five potentially suspicious KES 250,000 amounts. The amount and date issues will be investigated further during the cleaning process. The KES 250,000 values will be evaluated using the IQR method before deciding how they should be treated.

In [6]:
import pandas as pd

# Ensure txns is defined in the current namespace
if 'txns' not in locals() and 'txns' not in globals():
    txns = pd.read_csv("agent_transactions.csv")

duplicate_count = txns.duplicated(subset="txn_id").sum()

print("Duplicate transactions:", duplicate_count)

txns = txns.drop_duplicates(
    subset="txn_id",
    keep="first"
).copy()

print("Transactions after removing duplicates:", len(txns))

Duplicate transactions: 12
Transactions after removing duplicates: 600


In [7]:
# Check missing transaction types
missing_types = txns["txn_type"].isna().sum()

print("Missing transaction types:", missing_types)

# Standardise transaction labels
normalised_types = (
    txns["txn_type"]
    .astype("string")
    .str.strip()
    .str.lower()
)

txns["txn_type"] = normalised_types.map({
    "deposit": "Deposit",
    "withdrawal": "Withdrawal"
})

# Check results
print("\nStandardised transaction types:")
print(txns["txn_type"].value_counts(dropna=False))

print("\nMissing values after standardisation:")
print(txns["txn_type"].isna().sum())

Missing transaction types: 0

Standardised transaction types:
txn_type
Deposit       373
Withdrawal    227
Name: count, dtype: int64

Missing values after standardisation:
0


In [8]:
# Convert amount to numeric
txns["amount"] = pd.to_numeric(txns["amount"], errors="coerce")

# Convert transaction dates to datetime
txns["txn_date"] = pd.to_datetime(
    txns["txn_date"],
    format="mixed",
    errors="coerce"
)

# Check the results
print("AMOUNT DATA TYPE:")
print(txns["amount"].dtype)

print("\nMISSING AMOUNTS:")
print(txns["amount"].isna().sum())

print("\nDATE DATA TYPE:")
print(txns["txn_date"].dtype)

print("\nMISSING DATES:")
print(txns["txn_date"].isna().sum())

print("\nDATE RANGE:")
print(txns["txn_date"].min(), "to", txns["txn_date"].max())

AMOUNT DATA TYPE:
float64

MISSING AMOUNTS:
20

DATE DATA TYPE:
datetime64[ns]

MISSING DATES:
0

DATE RANGE:
2026-03-01 00:00:00 to 2026-04-29 00:00:00


In [9]:
# Record which amounts are missing before imputation
txns["amount_imputed"] = txns["amount"].isna()

# Calculate median amount for each transaction type
medians = txns.groupby("txn_type")["amount"].transform("median")

# Fill missing amounts using the relevant transaction type median
txns["amount"] = txns["amount"].fillna(medians)

# Verify the results
print("Missing amounts after imputation:", txns["amount"].isna().sum())

print("\nImputed transactions:")
print(txns["amount_imputed"].value_counts())

print("\nMedian amounts used:")
print(txns.groupby("txn_type")["amount"].median())

Missing amounts after imputation: 0

Imputed transactions:
amount_imputed
False    580
True      20
Name: count, dtype: int64

Median amounts used:
txn_type
Deposit       1803.5
Withdrawal    1750.5
Name: amount, dtype: float64


### Task 14: Handling Missing Transaction Amounts

There were 20 missing transaction amounts. I used median imputation separately for deposits and withdrawals to preserve all 600 transaction records. The median was selected because it is less sensitive to unusually large transaction values than the mean.

- Deposit median: KES 1,803.50
- Withdrawal median: KES 1,750.50

I created an `amount_imputed` Boolean column to identify the 20 records whose amounts were estimated. This maintains transparency and supports future data quality checks.

In [10]:
# Calculate the first and third quartiles
Q1 = txns["amount"].quantile(0.25)
Q3 = txns["amount"].quantile(0.75)

# Calculate the interquartile range
IQR = Q3 - Q1

# Calculate lower and upper limits
lower_limit = Q1 - 1.5 * IQR
upper_limit = Q3 + 1.5 * IQR

# Flag potential outliers
txns["is_outlier"] = (
    (txns["amount"] < lower_limit) |
    (txns["amount"] > upper_limit)
)

# Display results
print("Q1:", Q1)
print("Q3:", Q3)
print("IQR:", IQR)
print("Lower limit:", lower_limit)
print("Upper limit:", upper_limit)

print("\nPotential outliers:")
print(txns["is_outlier"].value_counts())

print("\nKES 250,000 transactions:")
print(txns[txns["amount"] == 250000][
    ["txn_id", "agent_id", "txn_type", "amount", "txn_date", "is_outlier"]
])

Q1: 1105.0
Q3: 2868.5
IQR: 1763.5
Lower limit: -1540.25
Upper limit: 5513.75

Potential outliers:
is_outlier
False    547
True      53
Name: count, dtype: int64

KES 250,000 transactions:
      txn_id agent_id    txn_type    amount   txn_date  is_outlier
306  TX00307    AG012     Deposit  250000.0 2026-03-02        True
315  TX00316    AG014     Deposit  250000.0 2026-04-01        True
424  TX00425    AG005  Withdrawal  250000.0 2026-03-02        True
463  TX00464    AG009  Withdrawal  250000.0 2026-04-11        True
594  TX00595    AG016     Deposit  250000.0 2026-03-11        True


### Task 15: Detecting Potential Outliers Using IQR

I used the Interquartile Range (IQR) method to identify unusually low or high transaction amounts.

- Q1: KES 1,105.00
- Q3: KES 2,868.50
- IQR: KES 1,763.50
- Upper limit: KES 5,513.75
- Potential outliers identified: 53

Five transactions had an amount of KES 250,000, and all five were flagged as potential outliers. These transactions should not be automatically deleted because they may represent legitimate high-value activity. They should be investigated against supporting transaction records before any correction or removal is considered.

In [11]:
# Merge transactions with the agent register
merged_txns = txns.merge(
    agents[["agent_id", "county"]],
    on="agent_id",
    how="left",
    indicator=True
)

# Check merge results
print("Merge results:")
print(merged_txns["_merge"].value_counts())

# Identify unknown agents
unknown_agents = merged_txns[
    merged_txns["_merge"] == "left_only"
]

print("\nUnknown agent transactions:", len(unknown_agents))

print("\nUnknown agent IDs:")
print(unknown_agents["agent_id"].value_counts())

# Check county missing values
print("\nMissing counties:", merged_txns["county"].isna().sum())

Merge results:
_merge
both          593
left_only       7
right_only      0
Name: count, dtype: int64

Unknown agent transactions: 7

Unknown agent IDs:
agent_id
AG099    7
Name: count, dtype: int64

Missing counties: 7


### Task 16: Merging Transactions with the Agent Register

I performed a left merge using `agent_id` to attach county information to each transaction.

The merge matched 593 transactions and identified 7 unmatched transactions belonging to agent `AG099`, which is absent from the agent register.

I retained all 600 transactions rather than deleting unmatched records. The county for AG099 remains missing until the agent's registration details can be verified. This preserves transaction history and maintains referential integrity transparency.

In [12]:
# Exclude unknown agents from county summaries because their county is unverified
known_county_txns = merged_txns[
    merged_txns["county"].notna()
].copy()

# County summary by transaction type
county_summary = (
    known_county_txns
    .groupby(["county", "txn_type"])
    .agg(
        total_value=("amount", "sum"),
        transaction_count=("txn_id", "count"),
        average_amount=("amount", "mean")
    )
    .reset_index()
)

print("COUNTY SUMMARY:")
display(county_summary)

COUNTY SUMMARY:


,county,txn_type,total_value,transaction_count,average_amount
0,Kiambu,Deposit,334737.0,43,7784.581395
1,Kiambu,Withdrawal,77024.5,27,2852.759259
2,Kisumu,Deposit,436501.5,74,5898.668919
3,Kisumu,Withdrawal,76348.5,37,2063.472973
4,Mombasa,Deposit,149094.0,72,2070.750000
5,Mombasa,Withdrawal,368592.0,61,6042.491803
6,Nairobi,Deposit,543698.0,118,4607.610169
7,Nairobi,Withdrawal,161932.0,62,2611.806452
8,Nakuru,Deposit,184944.0,60,3082.400000
9,Nakuru,Withdrawal,328568.5,39,8424.833333


In [13]:
# Create a weekly period column
known_county_txns["week"] = (
    known_county_txns["txn_date"]
    .dt.to_period("W")
    .astype(str)
)

# Calculate weekly transaction value by county
weekly_summary = (
    known_county_txns
    .groupby(["week", "county"])["amount"]
    .sum()
    .reset_index(name="total_value")
)

print("WEEKLY TRANSACTION VALUE BY COUNTY:")
display(weekly_summary)

WEEKLY TRANSACTION VALUE BY COUNTY:


,week,county,total_value
0,2026-02-23/2026-03-01,Kisumu,1893.0
1,2026-02-23/2026-03-01,Mombasa,6759.0
2,2026-02-23/2026-03-01,Nairobi,7046.0
3,2026-02-23/2026-03-01,Nakuru,2770.5
4,2026-03-02/2026-03-08,Kiambu,18317.0
5,2026-03-02/2026-03-08,Kisumu,287933.0
6,2026-03-02/2026-03-08,Mombasa,285696.5
7,2026-03-02/2026-03-08,Nairobi,68967.0
8,2026-03-02/2026-03-08,Nakuru,56574.0
9,2026-03-09/2026-03-15,Kiambu,267745.5


In [14]:
# Create weekly transaction pivot table

weekly_pivot = pd.pivot_table(
    known_county_txns,
    index="week",
    columns="county",
    values="amount",
    aggfunc="sum",
    fill_value=0
)

print("WEEKLY TRANSACTION PIVOT TABLE:")
display(weekly_pivot)

WEEKLY TRANSACTION PIVOT TABLE:


county,Kiambu,Kisumu,Mombasa,Nairobi,Nakuru
week,,,,,
2026-02-23/2026-03-01,0.0,1893.0,6759.0,7046.0,2770.5
2026-03-02/2026-03-08,18317.0,287933.0,285696.5,68967.0,56574.0
2026-03-09/2026-03-15,267745.5,23186.0,30504.5,52887.5,28730.0
2026-03-16/2026-03-22,22054.0,28126.0,25171.5,43134.5,39795.0
2026-03-23/2026-03-29,11504.0,27077.0,17050.0,41400.5,18972.5
2026-03-30/2026-04-05,22609.0,41077.0,38344.0,282931.0,18145.0
2026-04-06/2026-04-12,15030.0,15341.0,47119.5,56813.5,276609.0
2026-04-13/2026-04-19,12265.0,29470.0,26730.0,56406.5,26179.5
2026-04-20/2026-04-26,24507.0,37439.0,16687.0,65135.0,30862.0


In [15]:
# Import the cleaning function
from cleaning_pipeline import clean_agent_data

# Load the original raw datasets
raw_agents = pd.read_csv("agents.csv")
raw_txns = pd.read_csv("agent_transactions.csv")

# Run the reusable cleaning pipeline
clean_df, cleaning_log = clean_agent_data(
    raw_txns,
    raw_agents
)

# Display the cleaning log
print("CLEANING LOG:")
display(cleaning_log)

# Check the cleaned dataset
print("\nCLEANED DATASET SHAPE:")
print(clean_df.shape)

print("\nCLEANED DATASET PREVIEW:")
display(clean_df.head())

CLEANING LOG:


,action,rows_affected
0,Removed duplicate transaction IDs,12
1,Standardised transaction types,600
2,Imputed missing transaction amounts,20
3,Converted transaction dates,600
4,Flagged potential amount outliers,53
5,Merged agent county information,7



CLEANED DATASET SHAPE:
(600, 8)

CLEANED DATASET PREVIEW:


,txn_id,agent_id,txn_type,amount,txn_date,amount_imputed,is_outlier,county
0,TX00001,AG009,Deposit,353.0,2026-03-03,False,False,Nakuru
1,TX00002,AG020,Deposit,1238.0,2026-04-27,False,False,Mombasa
2,TX00003,AG008,Deposit,9129.0,2026-04-29,False,True,Kisumu
3,TX00004,AG009,Deposit,652.0,2026-04-24,False,False,Nakuru
4,TX00005,AG009,Withdrawal,478.0,2026-04-17,False,False,Nakuru


In [16]:
# Validate the cleaned dataset
print("DUPLICATE TRANSACTION IDs:")
print(clean_df["txn_id"].duplicated().sum())

print("\nTRANSACTION TYPES:")
print(clean_df["txn_type"].value_counts(dropna=False))

print("\nAMOUNT DATA TYPE:")
print(clean_df["amount"].dtype)

print("\nMISSING AMOUNTS:")
print(clean_df["amount"].isna().sum())

print("\nDATE DATA TYPE:")
print(clean_df["txn_date"].dtype)

print("\nMISSING DATES:")
print(clean_df["txn_date"].isna().sum())

print("\nIMPUTED AMOUNTS:")
print(clean_df["amount_imputed"].sum())

print("\nPOTENTIAL OUTLIERS:")
print(clean_df["is_outlier"].sum())

print("\nMISSING COUNTIES:")
print(clean_df["county"].isna().sum())

DUPLICATE TRANSACTION IDs:
0

TRANSACTION TYPES:
txn_type
Deposit       373
Withdrawal    227
Name: count, dtype: int64

AMOUNT DATA TYPE:
float64

MISSING AMOUNTS:
0

DATE DATA TYPE:
datetime64[ns]

MISSING DATES:
0

IMPUTED AMOUNTS:
20

POTENTIAL OUTLIERS:
53

MISSING COUNTIES:
7


In [17]:
# Validate the cleaned dataset
print("DUPLICATE TRANSACTION IDs:")
print(clean_df["txn_id"].duplicated().sum())

print("\nTRANSACTION TYPES:")
print(clean_df["txn_type"].value_counts(dropna=False))

print("\nAMOUNT DATA TYPE:")
print(clean_df["amount"].dtype)

print("\nMISSING AMOUNTS:")
print(clean_df["amount"].isna().sum())

print("\nDATE DATA TYPE:")
print(clean_df["txn_date"].dtype)

print("\nMISSING DATES:")
print(clean_df["txn_date"].isna().sum())

print("\nIMPUTED AMOUNTS:")
print(clean_df["amount_imputed"].sum())

print("\nPOTENTIAL OUTLIERS:")
print(clean_df["is_outlier"].sum())

print("\nMISSING COUNTIES:")
print(clean_df["county"].isna().sum())

DUPLICATE TRANSACTION IDs:
0

TRANSACTION TYPES:
txn_type
Deposit       373
Withdrawal    227
Name: count, dtype: int64

AMOUNT DATA TYPE:
float64

MISSING AMOUNTS:
0

DATE DATA TYPE:
datetime64[ns]

MISSING DATES:
0

IMPUTED AMOUNTS:
20

POTENTIAL OUTLIERS:
53

MISSING COUNTIES:
7


In [18]:
import os

# Create outputs folder if it does not exist
os.makedirs("outputs", exist_ok=True)

# Export cleaned transactions
clean_df.to_csv(
    "outputs/agent_transactions_clean.csv",
    index=False
)

print("Cleaned dataset exported successfully.")

Cleaned dataset exported successfully.


## Task 19: Data Quality Report — Privacy Considerations

If the transaction extract contained customer phone numbers, these would be considered personal data under Kenya's Data Protection Act, 2019, because they can identify or help identify individuals.

The organisation should:

1. Collect and process phone numbers only for a lawful and clearly defined purpose.
2. Limit access to authorised employees who need the information.
3. Protect the information through appropriate security measures.
4. Avoid exposing customer phone numbers in public reports, notebooks, or GitHub repositories.
5. Retain personal data only for as long as necessary and follow applicable retention requirements.

For this assignment, customer phone numbers should not be included in the public repository. Any reporting or analysis should use aggregated or anonymised information where possible.

These measures support responsible data handling and compliance with Kenya's Data Protection Act, 2019.

In [19]:
# Final data quality checks
print("Total rows:", len(clean_df))
print("Total columns:", len(clean_df.columns))

print("\nDuplicate transaction IDs:",
      clean_df["txn_id"].duplicated().sum())

print("\nTransaction types:")
print(clean_df["txn_type"].value_counts(dropna=False))

print("\nMissing amounts:", clean_df["amount"].isna().sum())
print("Missing dates:", clean_df["txn_date"].isna().sum())
print("Missing counties:", clean_df["county"].isna().sum())

print("\nImputed amounts:", clean_df["amount_imputed"].sum())
print("Potential outliers:", clean_df["is_outlier"].sum())

print("\nColumn data types:")
print(clean_df.dtypes)

Total rows: 600
Total columns: 8

Duplicate transaction IDs: 0

Transaction types:
txn_type
Deposit       373
Withdrawal    227
Name: count, dtype: int64

Missing amounts: 0
Missing dates: 0
Missing counties: 7

Imputed amounts: 20
Potential outliers: 53

Column data types:
txn_id                    object
agent_id                  object
txn_type                  object
amount                   float64
txn_date          datetime64[ns]
amount_imputed              bool
is_outlier                  bool
county                    object
dtype: object


In [20]:
from google.colab import files

files.download("outputs/agent_transactions_clean.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

### Data Privacy and Protection

Agent phone numbers and other personal information must be handled responsibly under Kenya's Data Protection Act, 2019. Personal information should only be collected and processed for a legitimate purpose, accessed by authorised personnel, and protected against unauthorised disclosure. Sensitive personal details should not be published in notebooks, reports or public GitHub repositories. Where possible, reports should use aggregated or anonymised data.